# Inferential Statistics - Manager Performance

## Purpose

This notebook tests whether manager portfolios differ on key outcomes using simple inferential models.
It complements the numeric and visualization manager notebooks by answering “are the differences likely real?” rather than just ranking.

Key questions:
1) Do average profit margins differ across managers?
2) Is return probability independent of manager portfolio?

Outputs:
- Hypothesis test + post-hoc comparisons for profit margin
- Logistic regression odds ratios for returns by manager
- Short interpretation after each result (no plots in this notebook)

## Contents
1. [Setup and data preparation](#1-setup-and-data-preparation)
2. [Profit margin differences by manager](#2-profit-margin-differences-by-manager)
3. [Return probability by manager](#3-return-probability-by-manager)
4. [Interpretation](#4-interpretation)

## 1. Setup and Data Preparation

In [1]:
# Load required libraries
library(tidyverse)
library(janitor)
library(dplyr)
library(ggplot2)
library(skimr)
library(purrr)
library(lubridate)

# Source helper scripts
source("../../R/apply_factors.R")
source("../../R/analysis_helpers.R")
source("../../R/temporal_helpers.R")

# Load data
tables <- list(
  Orders  = readr::read_csv("../../data/processed/Orders.csv"),
  Returns = readr::read_csv("../../data/processed/Returns.csv"),
  People  = readr::read_csv("../../data/processed/People.csv")
)

# Apply factor transformations
tables <- apply_factors(tables)

# Extract tables
orders  <- tables$Orders
returns <- tables$Returns
people  <- tables$People

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test


Rows: 51290 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (13): order_id, ship_mode, customer_name, segment, state, country, mark...
dbl   (6): sales, quantity, discount, profit, shipping_cost, year
date  (2): order_date, ship_date

ℹ Use `spec()` to retrieve the full column specification f

In [3]:
orders_mgr <- orders |>
    left_join(people |> rename(regional_manager = person), by = "region") |>
    mutate(
        regional_manager = if_else(is.na(regional_manager) | regional_manager == "", "Unassigned", regional_manager),
        margin = if_else(sales > 0, profit / sales, NA_real_)
    )

returns_one <- returns |>
    transmute(order_id, returned = as.integer(returned)) |>
    group_by(order_id) |>
    summarise(returned = as.integer(any(returned == TRUE)), .groups = "drop")

orders_mgr_ret <- orders_mgr |>
    left_join(returns_one, by = "order_id") |>
    mutate(returned = replace_na(returned, 0L))

## 2. Profit Margin Differences by Manager

### Model
We model profit margin as a function of the manager portfolio:

$$
\text{margin}_i \;=\; \mu \;+\; \alpha_{\text{manager}(i)} \;+\; \varepsilon_i
$$

Where:
- $\text{margin}_i$ is the profit margin for order $i$
- $\mu$ is the overall mean margin
- $\alpha_{\text{manager}(i)}$ is the effect of the manager associated with order $i$
- $\varepsilon_i$ is the error term

### Hypotheses
- **$H_0$:** All managers have the same mean margin  
  $$
  \alpha_1 = \alpha_2 = \cdots = \alpha_K = 0
  $$
  (equivalently: all manager mean margins are equal)

- **$H_1$:** At least one manager mean differs  
  $$
  \exists\, j \text{ such that } \alpha_j \neq 0
  $$

In [5]:
# ANOVA margin ~ manager
anova_margin_manager <- aov(margin ~ regional_manager, data = orders_mgr_ret)
summary(anova_margin_manager)

# Post-hoc: which managers differ (multiple-comparison controlled)
tuk_manager <- TukeyHSD(anova_margin_manager)
tuk_manager

                    Df Sum Sq Mean Sq F value Pr(>F)    
regional_manager    12    646   53.84   263.5 <2e-16 ***
Residuals        51277  10476    0.20                   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = margin ~ regional_manager, data = orders_mgr_ret)

$regional_manager
                                               diff          lwr           upr
Alejandro Ballentine-Unassigned         0.062320878  0.028227676  0.0964140804
Anna Andreadi-Unassigned                0.201407312  0.175961437  0.2268531874
Anthony Jacobs-Unassigned               0.224823628  0.191826923  0.2578203334
Chuck Magee-Unassigned                  0.195040506  0.167054490  0.2230265216
Deborah Brumfield-Unassigned           -0.002417171 -0.032988308  0.0281539651
Giulietta Dortch-Unassigned             0.222174077  0.180073535  0.2642746199
Jack Lebron-Unassigned                  0.266475361  0.236236824  0.2967138987
Kelly Williams-Unassigned               0.308457509  0.273342794  0.3435722248
Matt Collister-Unassigned               0.360717168  0.326867615  0.3945667204
Nicole Hansen-Unassigned                0.3887

### Interpretation 

The ANOVA tests whether mean profit margins are equal across manager portfolios.  
- If the p-value is small, we reject H0 and conclude that average margin differs for at least one manager.

Tukey’s post-hoc results identify which specific manager pairs differ while controlling for multiple comparisons.

Business meaning: Manager portfolios are not equally “efficient” in profitability terms. This should be interpreted as a portfolio/territory effect (region, market, customer/product mix), not as purely causal manager skill. Still, it is useful for identifying which portfolios warrant deeper diagnosis (discount intensity, shipping costs, returns, category mix).

## 3. Return Probability by Manager

### Model
We model return probability using a logistic regression with manager as a categorical predictor:

$$
\Pr(\text{returned}_i = 1) \;=\; \text{logit}^{-1}\!\left(\beta_0 \;+\; \beta_{\text{manager}(i)}\right)
$$

Equivalently, in log-odds form:

$$
\log\!\left(\frac{\Pr(\text{returned}_i = 1)}{1 - \Pr(\text{returned}_i = 1)}\right)
\;=\;
\beta_0 \;+\; \beta_{\text{manager}(i)}
$$

Where:
- $\text{returned}_i \in \{0,1\}$ indicates whether order $i$ was returned
- $\beta_0$ is the baseline log-odds for the reference (baseline) manager
- $\beta_{\text{manager}(i)}$ is the manager effect for the manager associated with order $i$

### Hypotheses
For each non-baseline manager coefficient:

- **$H_0$ (per coefficient):** Manager effect is zero (odds ratio = 1 vs baseline)  
  $$
  \beta_j = 0
  \quad \Longleftrightarrow \quad
  \exp(\beta_j) = 1
  $$

- **$H_1$:** Return odds differ by manager portfolio  
  $$
  \exists\, j \text{ such that } \beta_j \neq 0
  \quad \Longleftrightarrow \quad
  \exp(\beta_j) \neq 1
  $$

In [6]:
orders_mgr_ret <- orders_mgr_ret |>
  mutate(regional_manager = relevel(factor(regional_manager), ref = "Unassigned"))

return_manager_logit <- glm(
  returned ~ regional_manager,
  data = orders_mgr_ret,
  family = binomial(link = "logit")
)

summary(return_manager_logit)

# (odds ratios + 95% CI)
library(broom)
tidy(return_manager_logit, exponentiate = TRUE, conf.int = TRUE) |>
  filter(term != "(Intercept)") |>
  arrange(desc(estimate))


Call:
glm(formula = returned ~ regional_manager, family = binomial(link = "logit"), 
    data = orders_mgr_ret)

Coefficients:
                                       Estimate Std. Error z value Pr(>|z|)
(Intercept)                          -1.957e+01  1.516e+02  -0.129    0.897
regional_managerAlejandro Ballentine  1.660e+01  1.516e+02   0.109    0.913
regional_managerAnna Andreadi         1.672e+01  1.516e+02   0.110    0.912
regional_managerAnthony Jacobs        1.645e+01  1.516e+02   0.108    0.914
regional_managerChuck Magee           1.669e+01  1.516e+02   0.110    0.912
regional_managerDeborah Brumfield    -1.036e-08  2.196e+02   0.000    1.000
regional_managerGiulietta Dortch      1.630e+01  1.516e+02   0.107    0.914
regional_managerJack Lebron           1.762e+01  1.516e+02   0.116    0.907
regional_managerKelly Williams        1.668e+01  1.516e+02   0.110    0.912
regional_managerMatt Collister        1.785e+01  1.516e+02   0.118    0.906
regional_managerNicole Hansen       

Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”
Warning message:
“glm.fit: fitted probabilities numerically 0 or 1 occurred”

term,estimate,std.error,statistic,p.value,conf.low,conf.high
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
regional_managerShirley Daniels,66851693,151.6456,1.188164e-01,0.9054208,5.500373e+75,3.263609e+83
regional_managerMatt Collister,56778233,151.6456,1.177394e-01,0.9062741,4.671548e+75,2.771830e+83
regional_managerJack Lebron,45070396,151.6456,1.162166e-01,0.9074809,3.708255e+75,2.200267e+83
regional_managerAnna Andreadi,18187700,151.6456,1.102324e-01,0.9122251,1.496428e+75,8.878946e+82
regional_managerChuck Magee,17636873,151.6456,1.100296e-01,0.9123859,1.451114e+75,8.610082e+82
regional_managerKelly Williams,17600767,151.6456,1.100161e-01,0.9123966,1.448163e+75,8.592586e+82
regional_managerAlejandro Ballentine,16161962,151.6456,1.094537e-01,0.9128426,1.329779e+75,7.890167e+82
regional_managerAnthony Jacobs,13934163,151.6456,1.084756e-01,0.9136184,1.146481e+75,6.802574e+82
regional_managerGiulietta Dortch,11972170,151.6456,1.074749e-01,0.9144123,9.850834e+74,5.844956e+82


### 

This logistic regression estimates whether return likelihood differs across manager portfolios relative to the baseline manager.
- Exponentiated coefficients are **odds ratios**:
  - OR > 1 means higher return odds than baseline
  - OR < 1 means lower return odds than baseline
- If a 95% CI excludes 1, the difference is statistically significant.

Business meaning: Differences in return odds suggest structural regional differences in fulfillment quality, customer expectations, product mix, or shipping complexity. This is directly tied to profitability via reverse logistics and lost revenue. For managers with elevated odds, the next drill-down should be category/sub-category and ship mode within that portfolio.

## 4. Interpretation

- **Profit margins differ by manager portfolios** (ANOVA + Tukey): efficiency varies across regions managed by different individuals.
- **Return risk differs by manager portfolios** (logistic regression): some portfolios face systematically higher return likelihood.

Caveat:
Managers are associated with regions; these results should be interpreted as portfolio/territory differences (mix + structure), not purely causal manager ability.